In [82]:
import pandas as pd
import geopandas as gpd
import pycountry

In [83]:
gdf = gpd.read_file("original_data/countries.geo.json")

In [84]:
gdp_type = "PPP"

In [85]:
gdp_data = pd.read_csv(f"original_data/GDP_{gdp_type}_per_capita.csv")

In [86]:
year_cols = sorted([c for c in gdp_data.columns if isinstance(c, str) and c.isdigit() and len(c) == 4])

In [87]:
for c in gdp_data['REF_AREA_LABEL']:
  if c not in gdf['name'].values:
    print(c)

Aruba
Africa Eastern and Southern
Africa Western and Central
Andorra
Arab World
Antigua and Barbuda
Bahrain
Bahamas, The
Barbados
Brunei Darussalam
Central Electricity Board (CEB)
Cote d'Ivoire
Congo, Dem. Rep.
Congo, Rep.
Comoros
Cabo Verde
Caribbean small states
Curacao
Cayman Islands
Czechia
Dominica
East Asia & Pacific (excluding high income)
Early-demographic dividend
East Asia & Pacific
Europe & Central Asia (excluding high income)
Europe & Central Asia
Egypt, Arab Rep.
Euro area
European Union
Fragile and conflict affected situations
Faroe Islands
Micronesia, Fed. Sts.
Gambia, The
Guinea-Bissau
Grenada
High income
Hong Kong SAR, China
Heavily indebted poor countries (HIPC)
IBRD only
IDA & IBRD total
IDA total
IDA blend
IDA only
Iran, Islamic Rep.
Kyrgyz Republic
Kiribati
St. Kitts and Nevis
Korea, Rep.
Latin America & Caribbean (excluding high income)
Lao PDR
St. Lucia
Latin America & Caribbean
Least developed countries: UN classification
Low income
Lower middle income
Low & mid

In [88]:
for c in gdf['name']:
  if c not in gdp_data['REF_AREA_LABEL'].values:
    print(c)

Antarctica
French Southern and Antarctic Lands
The Bahamas
Brunei
Ivory Coast
Democratic Republic of the Congo
Republic of the Congo
Cuba
Northern Cyprus
Czech Republic
Egypt
Falkland Islands
Gambia
Guinea Bissau
French Guiana
Iran
Kyrgyzstan
South Korea
Laos
Macedonia
New Caledonia
North Korea
Russia
Western Sahara
Somaliland
Republic of Serbia
Slovakia
Swaziland
Syria
East Timor
Turkey
Taiwan
United Republic of Tanzania
United States of America
Venezuela
Vietnam
West Bank
Yemen


In [89]:
import country_converter as coco

gdp_data['iso3'] = coco.convert(names=gdp_data['REF_AREA_LABEL'].astype(str).tolist(), to='ISO3')
gdf['iso3'] = coco.convert(names=gdf['name'].astype(str).tolist(), to='ISO3')

Africa Eastern and Southern not found in regex
Africa Western and Central not found in regex
Arab World not found in regex
Central Electricity Board (CEB) not found in regex
Caribbean small states not found in regex
East Asia & Pacific ( not found in regex
Early-demographic dividend not found in regex
East Asia & Pacific not found in regex
Europe & Central Asia ( not found in regex
Europe & Central Asia not found in regex
Euro area not found in regex
European Union not found in regex
Fragile and conflict affected situations not found in regex
High income not found in regex
Heavily indebted poor countries (HIPC) not found in regex
IBRD only not found in regex
IDA & IBRD total not found in regex
IDA total not found in regex
IDA blend not found in regex
IDA only not found in regex
Latin America & Caribbean ( not found in regex
Latin America & Caribbean not found in regex
Least developed countries: UN classification not found in regex
Low income not found in regex
Lower middle income not f

In [90]:
for c in gdp_data['iso3']:
  if c not in gdf['iso3'].values:
    print(c)

ABW
not found
not found
AND
not found
ATG
BHR
BRB
not found
COM
CPV
not found
CUW
CYM
DMA
not found
not found
not found
not found
not found
not found
not found
not found
FRO
FSM
GRD
not found
HKG
not found
not found
not found
not found
not found
not found
KIR
KNA
not found
LCA
not found
not found
not found
not found
not found
not found
MAC
MDV
not found
MHL
not found
not found
MUS
not found
NRU
not found
not found
PLW
not found
not found
not found
not found
SGP
SMR
not found
not found
not found
STP
SXM
SYC
TCA
not found
not found
not found
not found
TON
not found
not found
TUV
not found
VCT
VIR
not found
WSM


In [91]:
for c in gdf['iso3']:
  if c not in gdp_data['iso3'].values:
    print(c)

ATA
ATF
CUB
FLK
GUF
NCL
PRK
ESH
TWN


In [92]:
def country_code_to_flag_emoji(code):
    if len(code) == 3:  # alpha-3
        try:
            country = pycountry.countries.get(alpha_3=code.upper())
            code2 = country.alpha_2
        except:
            return ""  # fallback if code not found
    elif len(code) == 2:  # alpha-2
        code2 = code.upper()
    else:
        return ""
    
    # convert letters to regional indicator symbols
    return "".join(chr(0x1F1E6 + ord(c) - ord('A')) for c in code2)

In [93]:
gdf["name_with_flag"] = gdf["iso3"].apply(country_code_to_flag_emoji) + " " + gdf["name"]

In [94]:
merged = gdf[['name', 'geometry', 'iso3', 'name_with_flag']].merge(gdp_data[year_cols + ['iso3']], on='iso3', how='left')
merged.to_file(f"transformed_data/GDP_{gdp_type}_per_capita.geojson")

In [95]:
df_years = merged.loc[:, [str(y) if str(y) in merged.columns else y for y in year_cols]]
gdp_growth = df_years.pct_change(axis=1) * 100
merged.loc[:, gdp_growth.columns] = gdp_growth

/tmp/ipykernel_5658/3755781085.py:2: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  gdp_growth = df_years.pct_change(axis=1) * 100


In [96]:
merged.to_file(f"transformed_data/{gdp_type}_Growth_rate.geojson")